# ⚙️ Lab 3 : Optimisation & Requêtes Planifiées

**Objectifs :**
- Réduire les coûts BigQuery avec le Partitionnement et le Clustering
- Comparer les octets scannés (Full Scan vs Partition Scan)
- Créer une Scheduled Query via le CLI

*(Rappel : Toujours travailler dans le dataset `thelook_prenom`)*

## Partie 1 : Partitionnement & Clustering
Pour éviter qu'une requête ne scanne toute la table (et coûte très cher), on découpe la table en "partitions" (généralement par date).

**Exemple de création de table partitionnée :**
```sql
CREATE OR REPLACE TABLE `thelook_prenom.silver_partitioned_logs`
PARTITION BY DATE(event_timestamp)
CLUSTER BY user_id
AS SELECT ...
```

### ✍️ À vous de jouer (Exercice 1)
**Consignes :**
1. Exécutez le code ci-dessous pour créer une table générant un million de logs avec une date aléatoire, partitionnée par `log_date` et clusterisée par `user_id`.
2. Exécutez ensuite les deux requêtes `SELECT COUNT(*)` (une avec filtre de date, l'autre sans) et observez la différence de **Bytes processed** en haut à droite de la console BigQuery.

In [ ]:
-- 1. Création de la table partitionnée et clustered
CREATE OR REPLACE TABLE `thelook_prenom.silver_web_logs`
PARTITION BY log_date
CLUSTER BY user_id AS
SELECT 
  DATE_ADD(CURRENT_DATE(), INTERVAL -CAST(FLOOR(RAND() * 30) AS INT64) DAY) AS log_date,
  CONCAT('USER_', CAST(FLOOR(RAND() * 1000) AS STRING)) AS user_id,
  'page_view' AS event_type
FROM UNNEST(GENERATE_ARRAY(1, 1000000));


In [ ]:
-- 2. Comparez le coût (Regardez "This query will process X MB")

-- Requête A (Sans filtre de date - Full Scan)
SELECT COUNT(*) FROM `thelook_prenom.silver_web_logs` WHERE user_id = 'USER_50';

-- Requête B (Avec filtre sur la partition - Partition Pruning)
SELECT COUNT(*) FROM `thelook_prenom.silver_web_logs` WHERE log_date = CURRENT_DATE() AND user_id = 'USER_50';


## Partie 2 : Requêtes Planifiées (Scheduled Queries)
Dans BigQuery, on peut automatiser une requête (ex: pour rafraîchir la couche Silver toutes les nuits). Sans DBT, c'est l'outil par défaut.

**Exemple via le terminal (`bq`) :**
```bash
bq query \
  --use_legacy_sql=false \
  --display_name="Daily Refresh Silver (prenom)" \
  --schedule="every 24 hours" \
  'CREATE OR REPLACE TABLE `mon_projet.thelook_prenom.silver_ma_table` AS SELECT * FROM `mon_projet.thelook_prenom.bronze_ma_table`;'
```

### ✍️ À vous de jouer (Exercice 2)
**Consignes :**
Dans un terminal (Cloud Shell ou votre VS Code), créez une requête planifiée qui va agréger les logs de `silver_web_logs` (nombre total d'événements par jour) et écrire le résultat dans `thelook_prenom.gold_daily_traffic` tous les jours à 02:00 du matin.

In [ ]:
# Écrivez votre commande bash `bq query` ici :

